# Spike: model-driven requirement-violation hunting

> **EXPERIMENTAL — this is a spike notebook, not a tutorial.** It is not
> wired into the docs, the tutorial suite, or the tests, and its executed
> outputs are committed deliberately (against repo hygiene) so it reads
> without running. It needs `hypothesis` and `allpairspy`, which are NOT
> project dependencies (scratch venv: `build/spikeenv`), plus the `[smt]`
> extra (z3). The helper prototype lives in
> `longeron.analysis._verify_spike`, clearly marked experimental.

**The premise.** A SysML v2 model already declares everything an
adversarial tester needs:

| the model says | a hunter reads it as |
| --- | --- |
| attribute types (`Real`, `Integer`, enums) | value **domains** → Hypothesis strategies |
| `assert constraint` bodies | **minable ranges** and pass/fail oracles |
| `assume` / `require` constraints | the universal property: *assumptions hold ⇒ requirements satisfied* |
| state machines (accepted events) | the **alphabet** of adversarial event sequences |
| variation catalogs | discrete spaces for **covering arrays** (NIST ACTS idea) |

Three hunters, one prey: **Hypothesis** samples continuous spaces and
*shrinks* what it catches; **covering arrays** squeeze discrete catalogs;
**Z3** *proves* presence or absence where the algebra is encodable. And
every catch materializes as an **M0 individual** — the concrete drone
that violates your requirement, inspectable like any other.

In [ ]:
import json
import time

import allpairspy
import hypothesis
import z3

import longeron
from longeron import m0
from longeron.analysis import _verify_spike as vs
from longeron.analysis.trades import TradeStudy

drone_model = longeron.load("../examples/drone.sysml")
uav_model = longeron.load("../examples/uav_missions.sysml")
di = longeron.Interpreter(drone_model)
ui = longeron.Interpreter(uav_model)
print("longeron", longeron.__version__ if hasattr(longeron, "__version__") else "(worktree)")
print(
    "hypothesis",
    hypothesis.__version__,
    "| allpairspy",
    allpairspy.__version__,
    "| z3",
    z3.get_version_string(),
)

## 1. Hypothesis × constraints: the universal property

`Interpreter.check_requirement` gives the property its exact shape — and
one semantic detail is **load-bearing**: a violated *assumption* makes a
requirement inapplicable (`applicable=False`, `satisfied=None`). That is
a **vacuous pass, never a failure**. The property is therefore

> for all configurations in the attribute domains:
> *every assumption holds* ⇒ *every `require` constraint (and every
> `assert constraint` on the subject) holds*.

The prey: `Drone::QuadCopter` from the demo model. Its free attribute is
`payloadMass`; `totalMass` derives from it; two `assert constraint`s
(`takeoffMassLimit`, `canHover`) and one requirement
(`FlightEnvelope::hoverMargin`, thrust-to-weight ≥ 1.8) sit on top.

First, the classic property-based-testing move, hand-mapped: a strategy
over `payloadMass ∈ [0, 5]` kg and a property targeting `canHover` alone.
Watch Hypothesis **shrink** the failure it finds down to the *simplest*
violating payload.

In [ ]:
from hypothesis import given, settings
from hypothesis import strategies as st


@settings(max_examples=200, derandomize=True, database=None)
@given(pm=st.floats(min_value=0.0, max_value=5.0, allow_nan=False))
def drone_can_hover(pm):
    drone = di.instantiate("Drone::QuadCopter", payloadMass=pm)
    can_hover = next(c for c in di.check(drone) if c.name == "canHover")
    assert can_hover.passed, (
        f"canHover violated: payloadMass={pm} kg -> totalMass="
        f"{drone.slots['totalMass']:.3f} kg (thrust 36.0 N < weight "
        f"{drone.slots['totalMass'] * 9.81:.2f} N)"
    )


try:
    drone_can_hover()
    print("no counterexample in 200 examples")
except AssertionError as err:
    print("FALSIFIED (shrunk):", err)

Shrinking hands back the *simplest* counterexample (a round 3.0 kg), not
the tightest one — by design: simple examples are the debuggable ones.
When the exact edge matters, refine the shrunk catch by bisection against
the same oracle, and compare with the closed form
(`36 N / 9.81 m/s² − 1.04 kg` of structure+battery+rotors):

In [ ]:
def violates_can_hover(pm: float) -> bool:
    drone = di.instantiate("Drone::QuadCopter", payloadMass=pm)
    return not next(c for c in di.check(drone) if c.name == "canHover").passed


edge = vs.bisect_boundary(violates_can_hover, 0.0, 3.0, tol=1e-9)
print(f"bisected boundary : payloadMass = {edge:.9f} kg")
print(f"closed form       : payloadMass = {36.0 / 9.81 - 1.04:.9f} kg")

### The full property, and the violating configuration

`_verify_spike.verdict` runs the whole property at one configuration —
all `assert constraint`s plus every requirement, vacuous-pass semantics
included — and `_verify_spike.hunt` wraps it in `hypothesis.find`, so the
returned counterexample is already **shrunk to the simplest bindings that
violate anything**. Here is the violating drone, shown clearly:

In [ ]:
worst = vs.hunt(
    di,
    "Drone::QuadCopter",
    requirements=("Drone::FlightEnvelope",),
    free=("payloadMass",),
    max_examples=200,
    fallback=(0.0, 5.0),
)
assert worst is not None
drone = di.instantiate("Drone::QuadCopter", **worst.bindings)
print("minimal violating configuration:")
print(f"  payloadMass = {worst.bindings['payloadMass']} kg")
print(
    f"  totalMass   = {drone.slots['totalMass']:.3f} kg "
    f"(maxTakeoffMass {drone.slots['maxTakeoffMass']} kg)"
)
print(f"  violated    : {worst.violated}")
print(f"  vacuous     : {worst.vacuous or 'none'}")
req = di.check_requirement("Drone::FlightEnvelope", subject=drone)
for c in req.requirements:
    print(f"  {req.name}::{c.name}: passed={c.passed}   [{c.expression}]")

### Auto-deriving the strategies from the model

The hand-mapped range above is the part that should come from the model.
`_verify_spike.attribute_domains` reads attribute **types** for the value
kind and **mines ranges** from constraint bodies that compare the
attribute directly against a literal. Two honest results:

- `QuadCopter.payloadMass` — **no direct bound exists in the model.** All
  constraints bind `totalMass`, which *derives from* `payloadMass`. The
  miner does not chase reachability through derived attributes (yet) —
  that fixed-point pass already exists in `analysis/smt.py`'s
  symbolic-marking loop and is the obvious next step for the design doc.
- `IsrPrime.loiterSpeed` (the UAV catalog's continuous sizing context) —
  its own `aboveStall`/`belowCruise` asserts yield **[11, 24] m/s**, no
  hand-mapping needed.

With the mined domain, the hunt is fully model-driven: strategy from the
model, property from the model, oracle from the interpreter.

In [ ]:
for interp, part, free in (
    (di, "Drone::QuadCopter", ("payloadMass",)),
    (ui, "UavMissions::IsrPrime", ("loiterSpeed",)),
):
    for name, dom in vs.attribute_domains(interp, interp.resolve(part), free).items():
        rng = f"[{dom.lo}, {dom.hi}]" if dom.bounded else "UNBOUNDED (fallback range)"
        src = f" mined from {dom.mined_from}" if dom.mined_from else ""
        print(f"{part}.{name}: {dom.kind} {rng}{src}")

In [ ]:
caught = vs.hunt(
    ui,
    "UavMissions::IsrPrime",
    requirements=("UavMissions::IsrStation",),
    free=("loiterSpeed",),
    max_examples=200,
)
assert caught is not None
uav = ui.instantiate("UavMissions::IsrPrime", **caught.bindings)
print(
    f"shrunk counterexample: loiterSpeed = {caught.bindings['loiterSpeed']} m/s "
    f"-> stationMinutes = {uav.slots['stationMinutes']:.1f} (floor: 90)"
)
print(f"violated: {caught.violated}")


def violates_station(v: float) -> bool:
    return not vs.verdict(
        ui, "UavMissions::IsrPrime", ("UavMissions::IsrStation",), {"loiterSpeed": v}
    ).ok


edge = vs.bisect_boundary(violates_station, 15.0, 24.0, tol=1e-6)
print(f"bisected edge: loiterSpeed = {edge:.4f} m/s is where the 90-minute station floor breaks")

## 2. Adversarial event sequences: `hypothesis.stateful` × state machines

A state machine gives the hunt a different shape: **rules** are the
events the machine accepts (read straight off its transitions), and the
**invariant** re-checks requirements after every step. Hypothesis then
searches *sequences*, and shrinks the catch to the minimal one.

The stock `Drone::FlightStates` has nothing sequence-violable (its only
variable monotonically counts launches), so this section adds a
spike-local model with a deliberately planted — and deliberately
*realistic* — bug: every climb-out burns 30% battery; the `launch` guard
enforces a 30% floor; but the **go-around path re-enters `airborne`
without re-checking the floor**. No single event violates anything. Only
a *sequence* does.

In [ ]:
SORTIE_SRC = """
package SpikeSortie {
    state def SortieStates {
        attribute battery : Integer := 100;

        entry; then idle;

        state idle;
        transition first idle accept launch if battery >= 30 then airborne;
        transition first idle accept recharge do assign battery := 100 then idle;

        state airborne {
            entry assign battery := battery - 30;   // each climb-out burns 30%
        }
        transition first airborne accept land then idle;
        // the bug: a go-around re-enters airborne, skipping the launch guard
        transition first airborne accept goAround then airborne;
    }
    requirement def BatteryNeverNegative {
        doc /* No sortie sequence may drive the pack below zero. */
        require constraint noDeepDischarge { battery >= 0 }
    }
}
"""
sortie_model = longeron.loads(SORTIE_SRC)
si = longeron.Interpreter(sortie_model)
sm_def = si.resolve("SpikeSortie::SortieStates")
EVENTS = vs.events_of(si, sm_def)
print("event alphabet (from the model):", EVENTS)

In [ ]:
from hypothesis import settings
from hypothesis import strategies as st
from hypothesis.stateful import (
    RuleBasedStateMachine,
    invariant,
    rule,
    run_state_machine_as_test,
)

from longeron.interpreter import StateMachine


class SortieHunt(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.sim = StateMachine(si, sm_def, {})
        self.sim.start()

    @rule(ev=st.sampled_from(EVENTS))
    def send(self, ev):
        self.sim.send(ev)  # non-matching events are recorded as ignored

    @invariant()
    def requirements_hold(self):
        env = dict(self.sim.env.frames[0])
        req = si.check_requirement("SpikeSortie::BatteryNeverNegative", bindings=env)
        assert req.satisfied is not False, (
            f"{req.name} violated: battery={env['battery']} in state {self.sim.current}"
        )


try:
    run_state_machine_as_test(
        SortieHunt,
        settings=settings(
            max_examples=100, derandomize=True, database=None, stateful_step_count=20
        ),
    )
    print("no violating sequence found")
except AssertionError as err:
    print("VIOLATING SEQUENCE FOUND:", err)
    print()
    for note in getattr(err, "__notes__", []):
        print(note)

The shrunk report is the pitch: `launch → goAround → goAround → goAround`
— **the minimal violating sortie**, with every irrelevant `land`/
`recharge`/ignored event stripped out by shrinking. The guard on `launch`
was correct; the model's *other* path into `airborne` was the hole. This
is exactly the class of bug requirement reviews miss and sequence hunts
find.

## 3. Covering arrays over the variation catalog

Discrete spaces want a different economy. The UAV catalog's ISR mission
has six variation points (4·3·3·3·3·2 = **648 mixes**) — still cheap to
enumerate exactly with the interpreter (that is `TradeStudy.
all_architectures`, our **ground truth**). A *pairwise covering array*
(the NIST ACTS idea, here via `allpairspy`) covers every **pair** of
variant choices in a handful of rows. The question a spike must answer
honestly: how many of the violations does the small array actually find?

In [ ]:
from allpairspy import AllPairs


def catalog_report(model, assembly):
    study = TradeStudy(model, assembly)
    names = list(study.points)
    domains = [sorted(study.points[n].variants) for n in names]
    t0 = time.time()
    exhaustive = study.all_architectures()
    t_exh = time.time() - t0
    pair_rows = [dict(zip(names, row, strict=True)) for row in AllPairs(domains)]
    t0 = time.time()
    pair_archs = [study.evaluate(r) for r in pair_rows]
    t_pair = time.time() - t0
    viol_exh, viol_pair = set(), set()
    for a in exhaustive:
        viol_exh.update(a.violations)
    for a in pair_archs:
        viol_pair.update(a.violations)
    print(f"{assembly}  (points: {'x'.join(str(len(d)) for d in domains)})")
    print(
        f"  exhaustive : {len(exhaustive):4d} mixes in {t_exh:5.2f}s | "
        f"infeasible {sum(not a.verified for a in exhaustive):4d} | "
        f"constraints violated somewhere: {len(viol_exh)}"
    )
    print(
        f"  pairwise   : {len(pair_archs):4d} mixes in {t_pair:5.2f}s | "
        f"infeasible {sum(not a.verified for a in pair_archs):4d} | "
        f"constraints caught: {len(viol_pair)}"
    )
    missed = viol_exh - viol_pair
    recall = len(viol_pair) / len(viol_exh) if viol_exh else 1.0
    print(
        f"  recall     : {len(viol_pair)}/{len(viol_exh)} violated "
        f"constraints ({recall:.0%})" + (f", MISSED: {sorted(missed)}" if missed else "")
    )
    print(f"  violated   : {sorted(viol_exh)}")
    return study, exhaustive, pair_archs


drone_cat = longeron.load("../examples/drone_catalog.sysml")
catalog_report(drone_cat, "DroneCatalog::TradeQuad")
print()
isr_study, isr_all, isr_pairs = catalog_report(uav_model, "UavMissions::IsrUav")

On both catalogs the pairwise array catches **every constraint that any
exhaustive mix violates** — at ~2.5% of the evaluations on the ISR
catalog. The honest caveats belong in the pitch too:

- pairwise *guarantees* pair coverage, **not** violation coverage — the
  100% recall here is measured (against feasible exhaustive ground
  truth), not promised. A violation needing three specific choices to
  line up can slip a 2-way array; that is what t-way (IPOG, t≥3) is for.
- the recall number is only *knowable* while exhaustive stays feasible.
  The array's real value begins exactly where the ground truth ends
  (7 points × 6 variants ≈ 280k mixes: enumeration minutes-to-hours,
  pairwise still ~50 rows).

## 4. The Z3 complement: proofs, exact bounds, honest refusals

Sampling can only ever say "found one" or "found none so far". For the
encodable subset of the model, `analysis/smt.py` says more: **is a
violation possible at all** (and here is the exact witness), or **provably
impossible**. Encode the quad with `payloadMass` free:

In [ ]:
from longeron.analysis import smt

sys_q = smt.to_smt(
    drone_model,
    "Drone::QuadCopter",
    requirements=("Drone::FlightEnvelope",),
    free=("payloadMass",),
)
print("assertions:", [label for label, _ in sys_q.assertions])
print("gaps      :", sys_q.gaps or "none")
result = sys_q.check()
print("\ncheck():", result.status)
print(
    "witness payloadMass =",
    result.witness["payloadMass"],
    "| totalMass =",
    result.witness["totalMass"],
)

That witness is a *finding*, not a glitch — two of them, in fact:

1. **`payloadMass = −1.04 kg` satisfies every encoded assertion.** The
   model never assumes payload mass is non-negative; Z3 exploits the gap
   instantly, where sampling over `[0, 5]` never would have looked.
2. The assumption that *should* have blocked the companion
   `totalMass = 0` witness — `FlightEnvelope`'s
   `assume constraint { drone.totalMass > 0.0 }` — **is missing from the
   assertion list** above. It is anonymous, and `smt.py`'s
   `named_members(...)` iteration silently drops unnamed constraints
   (the interpreter checks them fine; `check_requirement` proved it in
   §1). A real latent inconsistency between the two engines, caught by
   this spike — worth a ticket regardless of where the verify design
   goes.

Next, **exact feasibility bounds**. `maximize` with selective exclusion
isolates which constraint binds `payloadMass` where — as exact rationals,
with the strict inequality (`canHover` uses `>`) reported honestly as an
open bound (supremum − ε):

In [ ]:
sys_q2 = smt.to_smt(
    drone_model,
    "Drone::QuadCopter",
    requirements=("Drone::FlightEnvelope",),
    free=("payloadMass",),
)
nonneg = sys_q2.variables["payloadMass"] >= 0
sys_q2.assertions.append(("payloadMass.nonneg", nonneg))

rows = [
    ("takeoffMassLimit binds", ("QuadCopter::canHover", "FlightEnvelope::hoverMargin [require]")),
    ("hoverMargin binds", ("QuadCopter::takeoffMassLimit", "QuadCopter::canHover")),
    ("canHover binds", ("QuadCopter::takeoffMassLimit", "FlightEnvelope::hoverMargin [require]")),
]
print(f"{'binding constraint':28s} {'exact max payloadMass':>26s}")
for label, excludes in rows:
    bound, _ = sys_q2.maximize("payloadMass", exclude=excludes)
    print(f"{label:28s} {bound:>26s}")
print()
print(
    "compare: bisection in section 1 refined canHover to",
    f"{36.0 / 9.81 - 1.04:.9f} = 7166/2725 exactly; Z3 hands back the",
    "rational (minus epsilon: '>' is strict) by proof, not by search.",
)

And the crown move — a **proof of absence**. Assert the physics, the
non-negativity fix, and the *negation* of `hoverMargin`. Without the mass
budget the violation exists (SAT, exact witness); with `takeoffMassLimit`
asserted it is **UNSAT: no violating payload exists, proven** — the claim
no finite amount of sampling can make.

In [ ]:
def violation_possible(with_mass_budget: bool):
    solver = z3.Solver()
    for label, expr in sys_q.assertions:
        if label == "FlightEnvelope::hoverMargin [require]":
            solver.add(z3.Not(expr))
        elif label == "QuadCopter::takeoffMassLimit" and not with_mass_budget:
            continue
        elif label == "QuadCopter::canHover":
            continue  # isolate hoverMargin
        else:
            solver.add(expr)
    solver.add(sys_q.variables["payloadMass"] >= 0)
    status = solver.check()
    witness = ""
    if status == z3.sat:
        pm = solver.model().eval(sys_q.variables["payloadMass"])
        witness = f"  (witness payloadMass = {pm})"
    return status, witness


for budget in (False, True):
    status, witness = violation_possible(budget)
    label = "with   takeoffMassLimit" if budget else "without takeoffMassLimit"
    print(f"hoverMargin violation {label}: {status}{witness}")

Finally, the boundary of encodability itself — and it is sharper than
"nonlinear physics = no Z3". The ISR sizing context's station-time chain
runs through `pow`/`sqrt` momentum-theory physics, yet `smt.py`'s
constant-pinning (attributes unreachable from the *free* paths are pinned
to interpreter-exact values, never encoded) means **the same model is
encodable or not depending on where you free a variable**:

- free `loiterSpeed`: all the `pow`/`sqrt` physics sits *upstream* of it,
  gets pinned, and never hits the solver — Z3 takes the query whole and
  can even bracket the edge that §1's bisection found by search;
- free `emptyMassKg`: the hover-power `pow` chain now *depends on* the
  free variable — and the builder refuses honestly, recording the gap
  instead of pretending.

In [ ]:
sys_isr = smt.to_smt(
    uav_model,
    "UavMissions::IsrPrime",
    requirements=("UavMissions::IsrStation",),
    free=("loiterSpeed",),
)
print("free loiterSpeed  -> gaps:", sys_isr.gaps or "none (physics pinned upstream)")


def station_violation_possible(*extra):
    solver = z3.Solver()
    for label, expr in sys_isr.assertions:
        solver.add(z3.Not(expr) if label == "IsrStation::stationFloor [require]" else expr)
    for expr in extra:
        solver.add(expr)
    return solver.check()


v = sys_isr.variables["loiterSpeed"]
print("  stationFloor violation within [11, 24] m/s :", station_violation_possible())
print(
    "  ...with loiterSpeed <= 19.0 m/s            :",
    station_violation_possible(v <= 19.0),
    "<- proof of absence",
)
print("  ...with loiterSpeed <= 19.5 m/s            :", station_violation_possible(v <= 19.5))
print("  (section 1's bisection put the edge at 19.4790 m/s -- Z3 just")
print("   bracketed it by proof, physics pinned, no sampling)")

sys_isr2 = smt.to_smt(
    uav_model,
    "UavMissions::IsrPrime",
    requirements=("UavMissions::IsrStation",),
    free=("emptyMassKg",),
)
print("\nfree emptyMassKg -> the honest refusal:")
for gap in sys_isr2.gaps:
    print("  -", gap)

**Division of labor, measured on this spike:**

| model shape | winner | what you get |
| --- | --- | --- |
| linear-ish algebra (`QuadCopter`, `TradeQuad` rules) | **Z3** | exact bounds, unsat cores, *proof of absence* |
| nonlinear physics *upstream* of the free variables (`IsrPrime.loiterSpeed`) | **Z3** (pinning) | still proof — the physics never hits the solver |
| nonlinear physics *reached by* the free variables (`IsrPrime.emptyMassKg`, mission trades) | **Hypothesis** | shrunk counterexamples, boundary bisection |
| discrete catalogs, big products | **covering arrays** | pair coverage at ~2–5% of exhaustive cost |
| discrete catalogs, small products | exhaustive (`trades`) | the recall ground truth, interpreter-exact |
| event-driven behavior | **hypothesis.stateful** | minimal violating event sequences |

## 5. The catch, materialized: M0 individuals

A counterexample that lives in a test log dies in a test log. `longeron.
m0` turns the catch into a **population of identified individuals** — the
concrete drone that violates the requirement, with stable ids,
inspectable in the explorer and red on the scoreboard like any other
interpretation.

First the §1 catch (`payloadMass = 1.0`), as an M0 interpretation:

In [ ]:
bad_drone = m0.interpret(drone_model, "Drone::QuadCopter", bindings={"payloadMass": 1.0})
print("individuals:")
for ind in bad_drone.individuals():
    print("  ", ind.id)
print()
for con in di.check(bad_drone.root):
    verdict_str = "PASS" if con.passed else "**VIOLATED**"
    print(f"  {con.name:18s} {verdict_str}   [{con.expression}]")
req = di.check_requirement("Drone::FlightEnvelope", subject=bad_drone.root)
print(f"  FlightEnvelope     applicable={req.applicable} satisfied={req.satisfied}")
print()
print(
    json.dumps(
        {k: v for k, v in bad_drone.root.to_dict().items() if not isinstance(v, (dict, list))},
        indent=2,
    )
)

And a covering-array catch from §3: the violating *architecture* becomes
a partial interpretation via `m0.from_architecture` — variant selections
pinned, multiplicities populated, every individual carrying the selected
variant's real values:

In [ ]:
bad_mix = next(a for a in isr_pairs if not a.verified)
print("violating mix :", bad_mix.selection)
print("violations    :", bad_mix.violations)
mix_m0 = m0.from_architecture(isr_study, bad_mix)
print("root          :", mix_m0.root.id)
print("selection     :", mix_m0.selection)
motors = mix_m0.root.slots["motors"]
print(
    f"motors        : {[m.id.split('.')[-1] for m in motors]}"
    f" (maxThrust {motors[0].slots['maxThrust']} N each)"
)
print(
    f"rollup        : sum(motors.mass) = {mix_m0.rollup('sum(motors.mass)'):.3f} kg"
    " over the four real individuals"
)
print(
    f"stationMinutes: {bad_mix.metrics['stationMinutes']:.1f}"
    " (trades-exact; floor 25.0) -> the red scoreboard cell"
)
print(
    f"gaps          : {len(mix_m0.gaps)} M1 expressions that lean on the"
    " homogeneous '4.0 * x' convention degrade honestly over a real"
    " heterogeneous-capable population, e.g.:"
)
print("   ", mix_m0.gaps[0])

## The pitch, and what a real `longeron.analysis.verify` would be

What this spike demonstrated end to end, in under a minute of compute:

1. **Property + shrinking** — the universal property falls out of
   `check_requirement`'s existing semantics (vacuous passes included);
   Hypothesis finds violations and shrinks them to the simplest
   configuration; bisection refines the edge.
2. **Strategies from the model** — types give domains, constraints give
   ranges (`IsrPrime` needed zero hand-mapping); the known gap
   (reachability through derived attributes) already has its prototype
   in `smt.py`'s fixed point.
3. **Sequence hunting** — rules read off the transitions, invariants =
   requirement checks, and shrinking returns the *minimal* violating
   sortie.
4. **Covering arrays** — measured 100% violated-constraint recall at
   ~2.5% of exhaustive cost on the ISR catalog, with the honest caveat
   attached.
5. **Z3** — exact bounds, a proof of absence, one genuine model gap
   (negative payload) and one genuine tool gap (anonymous `assume`
   dropped by the encoder) found by proof.
6. **M0 materialization** — every catch becomes the inspectable
   individual that violates the requirement.

API sketch (design doc, not this spike):

```python
from longeron.analysis import verify

report = verify.hunt(model, "Drone::QuadCopter",          # Hypothesis
                     requirements=("Drone::FlightEnvelope",),
                     free=("payloadMass",))                # or auto-derived
report = verify.sequences(model, "Drone::FlightStates",    # stateful
                          requirements=(...))
report = verify.pairwise(model, "UavMissions::IsrUav")     # covering array
report = verify.prove(model, "Drone::QuadCopter",          # Z3 (encodable)
                      requirements=(...), free=(...))

report.counterexamples[0].materialize()   # -> m0.Interpretation (red on
                                          #    the scoreboard, in the explorer)
```

See `spike_verify_FINDINGS.md` next to this notebook for the full
findings, feasibility notes, and recommended design-doc scope.